# Session 2 — RAG Fundamentals

## What is RAG, and why

An LLM only knows what was in its training data — nothing about your private docs, nothing after its
training cutoff. **Retrieval-Augmented Generation (RAG)** fixes this without retraining the model: at
query time, fetch the most relevant pieces of *your* data and stuff them into the prompt, so the model
generates its answer grounded in that context instead of guessing from memory.

The core pipeline, end to end:

`docs -> chunks -> embeddings -> vector store -> similarity search -> augmented prompt -> generation`

We'll build every piece from scratch with raw numpy first, then show the same pipeline in LangChain.

In [1]:
import os
import numpy as np
from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

EMBED_MODEL = "gemini-embedding-001"
CHAT_MODEL = "gemini-2.5-flash-lite"

## Embeddings from scratch

A text **embedding** is a fixed-length vector of floats that captures the *meaning* of a piece of text.
Semantically similar text ends up as vectors that point in similar directions. Let's embed a few
sentences and look at the shape.

In [2]:
sentences = [
    "The cat sat on the mat.",
    "A feline rested on the rug.",
    "The stock market fell sharply today.",
]

def embed(texts):
    resp = client.models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in resp.embeddings])

vectors = embed(sentences)
vectors.shape  # (3, embedding_dim) -- one vector per sentence

(3, 3072)

## Similarity — cosine similarity by hand

Cosine similarity measures the angle between two vectors, ignoring magnitude: 1.0 means identical
direction (very similar meaning), 0 means unrelated, -1 means opposite. It's the standard measure for
comparing embeddings.

In [3]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# sentence 0 (cat) vs sentence 1 (feline) -- should score high, both about a cat resting
print("cat vs feline:", cosine_similarity(vectors[0], vectors[1]))

# sentence 0 (cat) vs sentence 2 (stock market) -- unrelated topics, should score low
print("cat vs stock market:", cosine_similarity(vectors[0], vectors[2]))

cat vs feline: 0.7705667422467881
cat vs stock market: 0.5800395807953744


## Chunking documents

Real documents are too long to embed as one blob and too long to stuff whole into a prompt. We split
them into **chunks** — small enough to embed precisely, big enough to keep context coherent. Chunk size
is a tradeoff: too small loses surrounding context, too large drowns out the relevant part and wastes
prompt budget. We'll use a simple fixed-size character splitter with overlap.

In [4]:
def chunk_text(text, chunk_size=200, overlap=40):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]

sample_doc = """
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a
language model. Instead of relying solely on what the model learned during training, RAG fetches
relevant passages from an external knowledge base at query time and feeds them to the model as
context. This lets the model answer questions about private, recent, or niche information it was
never trained on, without needing to fine-tune or retrain it. The retrieval step typically uses
vector similarity search over document embeddings.
"""

chunks = chunk_text(sample_doc)
len(chunks), chunks[0]

(4,
 'Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a\nlanguage model. Instead of relying solely on what the model learned during training, RAG fetches\nrelevant p')

## Building the in-memory vector store

Embed every chunk once, stack the vectors into a matrix, and keep the matrix in memory alongside the
original chunk text. To query, embed the question and compare it against every row with cosine
similarity — a `top_k` scan is all a small in-memory store needs.

In [5]:
class VectorStore:
    def __init__(self, chunks):
        self.chunks = chunks
        self.vectors = embed(chunks)

    def query(self, question, top_k=2):
        q_vec = embed([question])[0]
        sims = np.array([cosine_similarity(q_vec, v) for v in self.vectors])
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.chunks[i], sims[i]) for i in top_idx]

store = VectorStore(chunks)

## Retrieval

Given a user question, embed it and pull back the top-k most similar chunks.

In [6]:
results = store.query("How does RAG help a model answer questions about private data?")
for chunk, score in results:
    print(f"{score:.3f} -> {chunk[:80]}...")

0.842 -> during training, RAG fetches
relevant passages from an external knowledge base a...
0.768 -> Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval sy...


## Augmented prompt + generation

Stuff the retrieved chunks into a prompt template as context, then call the model. The model now
answers grounded in the retrieved text instead of purely from its training data.

In [7]:
def rag_answer(question, store, top_k=2):
    retrieved = store.query(question, top_k=top_k)
    context = "\n\n".join(chunk for chunk, _ in retrieved)
    prompt = f"""Answer the question using only the context below.

Context:
{context}

Question: {question}"""

    resp = client.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return resp.text

print(rag_answer("How does RAG help a model answer questions about private data?", store))

RAG helps a model answer questions about private data by fetching relevant passages from an external knowledge base at query time and feeding them to the model as context.


## End-to-end demo

A tiny multi-document corpus, and a question the base model has no way of knowing the answer to
(it's about a fictional internal tool). Watch retrieval find the one relevant doc and ground the
answer in it.

In [8]:
corpus = [
    "The Nimbus deploy tool ships code to production every 15 minutes via a cron-triggered pipeline.",
    "Photosynthesis converts light energy into chemical energy stored in glucose.",
    "The French Revolution began in 1789 and led to the end of the monarchy.",
    "Nimbus requires a green CI run and one human approval before any deploy proceeds.",
]

all_chunks = [c for doc in corpus for c in chunk_text(doc, chunk_size=200, overlap=0)]
demo_store = VectorStore(all_chunks)

question = "What has to happen before Nimbus deploys code?"
print(rag_answer(question, demo_store))

Before Nimbus deploys code, a green CI run and one human approval must happen.


## Same pipeline with LangChain

LangChain packages every piece above as a reusable component: a text splitter, an embeddings wrapper,
a vector store with a `.as_retriever()` interface, and a chain that wires retrieval + prompt +
generation together. Same pipeline, less hand-rolled plumbing.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)
lc_chunks = splitter.split_text("\n".join(corpus))

lc_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=api_key)
lc_store = InMemoryVectorStore.from_texts(lc_chunks, embedding=lc_embeddings)
retriever = lc_store.as_retriever(search_kwargs={"k": 2})

lc_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
)
lc_model = ChatGoogleGenerativeAI(model=CHAT_MODEL, google_api_key=api_key)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | lc_prompt
    | lc_model
    | StrOutputParser()
)

print(rag_chain.invoke("What has to happen before Nimbus deploys code?"))

A green CI run and one human approval.
